# 04 — Effect Size & Power Analysis

Statistical significance tells you *whether* an effect exists. Effect size and power tell you *how big* it is and *how reliably* you can detect it.

---
**What this notebook covers:**
- Cohen's h for the arcsine-transformed difference in proportions
- Absolute vs relative lift as practical metrics
- Observed power for the current sample size
- Pre-experiment sample size planning using MDE

In [ ]:
# ── path setup — run this cell first ─────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve().parent
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
from ab_testing_framework.effect_size import calculate_effect_size
from ab_testing_framework.validation import validate_input

experiment = validate_input(10000, 450, 10000, 520)
es = calculate_effect_size(experiment)

print(f'Absolute difference:  {es.absolute_difference:.4%}')
rel = f'{es.relative_improvement:.2%}' if es.relative_improvement is not None else 'N/A'
print(f'Relative improvement: {rel}')
print(f"Cohen's h:            {es.cohens_h:.4f}")

In [ ]:
# Cohen's h interpretation (conventional thresholds)
h = abs(es.cohens_h)
if h < 0.2:
    magnitude = 'small'
elif h < 0.5:
    magnitude = 'medium'
else:
    magnitude = 'large'

print(f"Cohen's h = {h:.4f} → {magnitude} effect")
print('(Conventional thresholds: small < 0.2, medium < 0.5, large ≥ 0.5)')

In [ ]:
from ab_testing_framework.power_analysis import analyze_power

pa = analyze_power(10000, 450, 10000, 520, alpha=0.05, target_power=0.80)

print(f'Observed power:            {pa.power:.1%}')
print(f'Required n / group (80%):  {pa.required_sample_size_per_group:,}')
print(f'Current n (control):       {experiment.visitors_a:,}')
adequate = experiment.visitors_a >= pa.required_sample_size_per_group
print(f'Sample adequately powered: {adequate}')

In [ ]:
# Pre-experiment planning: how many visitors do I need?
from ab_testing_framework.power_analysis import estimate_sample_size

baseline   = 0.045   # current conversion rate
mde_values = [0.001, 0.003, 0.005, 0.010]  # minimum detectable effects

print(f'Baseline rate: {baseline:.1%}')
print(f'{"MDE":>8}  {"Expected rate":>14}  {"n / group":>10}')
print('-' * 38)
for mde in mde_values:
    expected = baseline + mde
    if expected <= 0 or expected >= 1:
        continue
    n = estimate_sample_size(baseline_rate=baseline, expected_rate=expected,
                             alpha=0.05, target_power=0.80)
    print(f'{mde:>+8.3%}  {expected:>14.3%}  {n:>10,}')

### Interpretation

- **Cohen's h ≈ 0.034** — a small effect by conventional thresholds, but meaningful in practice for conversion rate optimisation where even a 1 pp lift on high-traffic pages compounds significantly.
- **Observed power > 80%** — the experiment was adequately powered to detect the observed effect reliably.
- The **sample size planning table** shows how much data you need depending on the minimum detectable effect you care about. Smaller MDEs require larger samples.
- Always determine your MDE *before* running the experiment. Powering for what you already saw (post-hoc power analysis) is circular and should be avoided for planning decisions.